<a href="https://colab.research.google.com/github/aditya0589/notebooks/blob/main/AI%20Engineering/Langchain/LC04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LC04 - Structured Output**

In Langchain, structured output refers to the practice of having language models return responses in well-defined data format (like JSON), rather than in free form text. This makes the model output easier to parse and work with programmatically.

PROMPT
------
"Can you create a one-day travel itinerary for Paris?"

UNSTRUCTURED LLM RESPONSE
-------------------------
Here’s a suggested itinerary:
- Morning: Visit the Eiffel Tower.
- Afternoon: Walk through the Louvre Museum.
- Evening: Enjoy dinner at a Seine riverside café.

Explanation:
This response is written in natural language. While it is easy for humans to read, it is unstructured, which makes it harder for programs to reliably parse, validate, or reuse the information (for example, in a UI, API response, or database).

JSON-ENFORCED OUTPUT
--------------------
[<br>
  {<br>
    "time": "Morning",<br>
    "activity": "Visit the Eiffel Tower"<br>
  },<br>
  {<br>
    "time": "Afternoon",<br>
    "activity": "Walk through the Louvre Museum"<br>
  },<br>
  {<br>
    "time": "Evening",<br>
    "activity": "Enjoy dinner at a Seine riverside café"<br>
  }<br>
]



Here, the same information is represented in a structured JSON format.
- Each itinerary item is an object with fixed keys ("time" and "activity").
- The overall output is an array, making it predictable and machine-readable.
- This format is ideal for APIs, frontend rendering, validation, and downstream automation.

Key Idea:
Unstructured text is human-friendly.
JSON-enforced output is machine-friendly, consistent, and reliable for production systems.


---------------------------------------------
## **Why do we need Structured output**

1. Data Extraction
2. API building
3. Agents

---------------------------

There are some LLMs which can generate structured outputs and some which cannot.

For the models which CAN generate structured output, we can use the langchain `` with_structured_output`` function to get the data in whatever output format we require.

For the models which CANNOT generate structured output, we must use output parsers.

## **with_structured_output function**:

We need to call the function and specify the data format before invoking the model.

There are three major data formats which you can use.

1. TypedDict
2. Pydantic
3. json_schema


## **TypedDict**
TypedDict is a way to represent a dictionary in Python where you specify what keys and values should exist. It helps ensure that your dictionary follows a specific structure.

Why use TypedDict:
1. It tells python what keys are required and what kind of values they should have.
2. It does not validate data at runtime.


In [2]:
from typing import TypedDict

class Book(TypedDict):
    title: str
    author: str
    year: int
my_book: Book = {
    "title": "Mein kampf",
    "author": "Adolf Hitler",
    "year": 1925
}

print(my_book)

# You can access elements like a regular dictionary
print(f"Title: {my_book['title']}")
print(f"Author: {my_book['author']}")

# This will raise a type-checking error (but not a runtime error) because 'rating' is not defined in Book
# my_book_invalid: Book = {
#     "title": "Another Book",
#     "author": "Another Author",
#     "year": 2000,
#     "rating": 5 # This line would be flagged by a type checker
# }

# This will raise a type-checking error (but not a runtime error) if 'title' is missing
# my_book_incomplete: Book = {
#     "author": "Someone",
#     "year": 2023
# }

{'title': 'Mein kampf', 'author': 'Adolf Hitler', 'year': 1925}
Title: Mein kampf
Author: Adolf Hitler


In [3]:
!pip install langchain langchain_community langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [1]:
import os
from google.colab import userdata

# Load keys from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGINGFACEHUB_API_TOKEN'] = userdata.get('HUGGINGFACEHUB_API_TOKEN')

# Sanity check (safe partial print)
print("Groq:", os.environ["GROQ_API_KEY"][:8])
print("Google:", os.environ["GOOGLE_API_KEY"][:8])
print('HF_TOKEN', os.environ['HF_TOKEN'][:8])
print('huggingface api token', os.environ['HUGGINGFACEHUB_API_TOKEN'][:8])

Groq: gsk_L243
Google: AIzaSyCX
HF_TOKEN AIzaSyBE
huggingface api token hf_SMPRD


Let us consider a use case on how we can use the TypedDict in LLMs

In [2]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional, Literal

load_dotenv()

model = ChatGroq(model_name="llama-3.1-8b-instant")

# schema
class Review(TypedDict):

    key_themes: Annotated[list[str], "Write down all the key themes discussed in the review in a list"]
    summary: Annotated[str, "A brief summary of the review"]
    sentiment: Annotated[Literal["pos", "neg"], "Return sentiment of the review either negative, positive or neutral"]
    pros: Annotated[Optional[list[str]], "Write down all the pros inside a list"]
    cons: Annotated[Optional[list[str]], "Write down all the cons inside a list"]
    name: Annotated[Optional[str], "Write the name of the reviewer"]


structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful

Review by Nitish Singh
""")
print(result)
print(result['name'])

{'cons': ['Uncomfortable weight and size for one-handed use', 'Bloatware in Samsung One UI', 'Expensive price tag'], 'key_themes': ['Performance', 'Camera', 'Battery Life'], 'name': 'Nitish Singh', 'pros': ['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'], 'sentiment': 'pos', 'summary': 'The Samsung Galaxy S24 Ultra is a powerhouse device with great performance, camera, and battery life, but it has its drawbacks.'}
Nitish Singh


## **Pydantic:**

Pydantic is a data validation and data parsing liberary for python. It ensures that the data you work with is correct, structured and type-safe.

In [4]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field

load_dotenv()

model = ChatGroq(model_name="llama-3.1-8b-instant")

# schema
class Review(BaseModel):

    key_themes: list[str] = Field(description="Write down all the key themes discussed in the review in a list")
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg"] = Field(description="Return sentiment of the review either negative, positive or neutral")
    pros: Optional[list[str]] = Field(default=None, description="Write down all the pros inside a list")
    cons: Optional[list[str]] = Field(default=None, description="Write down all the cons inside a list")
    name: Optional[str] = Field(default=None, description="Write the name of the reviewer")


structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful

Review by Nitish Singh
""")

print(result)

key_themes=['processor performance', 'camera quality', 'battery life', 'S-Pen support', 'bloatware', 'price'] summary='The Samsung Galaxy S24 Ultra is a powerful device with a stunning camera, long battery life, and unique S-Pen support, but it has some drawbacks like weight, bloatware, and a high price tag.' sentiment='pos' pros=['insanely powerful processor', 'stunning 200MP camera', 'long battery life', 'S-Pen support'] cons=['weight and size', 'bloatware', 'price'] name='Nitish Singh'


## **JSON**

In [5]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Optional, Literal
from pydantic import BaseModel, Field

load_dotenv()

model = ChatGroq(model_name = "llama-3.1-8b-instant")

# schema
json_schema = {
  "title": "Review",
  "type": "object",
  "properties": {
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write down all the key themes discussed in the review in a list"
    },
    "summary": {
      "type": "string",
      "description": "A brief summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["pos", "neg"],
      "description": "Return sentiment of the review either negative, positive or neutral"
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the pros inside a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the cons inside a list"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer"
    }
  },
  "required": ["key_themes", "summary", "sentiment"]
}


structured_model = model.with_structured_output(json_schema)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful

Review by Nitish Singh
""")

print(result)

{'cons': ['Weight and size make it uncomfortable for one-handed use', 'Bloatware from Samsung', 'High price tag of $1,300'], 'key_themes': ['powerful processor', 'camera', 'battery life', 'S-Pen', 'weight', 'size', 'bloatware', 'price'], 'name': 'Nitish Singh', 'pros': ['Insanely powerful processor', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful'], 'sentiment': 'pos', 'summary': 'The Samsung Galaxy S24 Ultra is a powerhouse with a great processor, camera, and battery life, but has some drawbacks such as its weight, size, and bloatware.'}


## **Output Parsers**

Output parsers in Langchain help convert raw LLM responses into structured formats like JSON, CSV, Pydantic models and more. They ensure consistancy, validation and ease of use in applications.

### **StrOutputParser**
The StrOutputParser is the simplest output parser in langchain. It is used to parse the output of a large language model (LLM) and return it as a plain string.

**WHY WOULD YOU USE THIS PARSER WHEN THE LLM OUTPUT ITSELF IS A STRING?**

Lets look at asimple task.

We want to give a topic to an LLM and ask it to summarize it. Then give this summary to an other LLM and ask it to summarize it in exactly 5 lines.

Lets look at this problem both using the normal approach and by using StrOutputParser

In [11]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

model = ChatGroq(
    model_name = "llama-3.1-8b-instant",
)

# prompt 01
template01 = PromptTemplate(
    template = "Write a detailed report on {topic}",
    input_variables = ['topic']
)

# prompt 02
template02 = PromptTemplate(
    template = "Write a 5 point summary on the following report {report}",
    input_variables = ['report']
)

prompt1 = template01.invoke({'topic': 'black hole'})

result = model.invoke(prompt1)

prompt2 = template02.invoke({'report': result})
result02 = model.invoke(prompt2)
print(result02.content)


Here's a 5-point summary of the report content on Black Holes:

1. **Formation of Black Holes**: Black holes are formed when a massive star runs out of fuel and dies, causing it to collapse under its own gravity. This compression creates an intense gravitational field that warps spacetime around the star, forming a boundary called the event horizon. Once matter crosses the event horizon, it is trapped by the black hole's gravity and cannot escape.

2. **Characteristics of Black Holes**: Black holes have several unique characteristics, including mass (determined by the star that formed them), spin (which affects matter behavior), charge (although rare), event horizon (beyond which nothing can escape), and singularity (the point at the center where gravity is infinite).

3. **Types of Black Holes**: There are four main types of black holes: stellar black holes (formed from individual stars), intermediate-mass black holes (100-100,000 solar masses), supermassive black holes (millions or b

Now lets do the same thing using StrOutputParser

In [12]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [13]:
chain = template01 | model | parser | template02 | model | parser

In [15]:
chain.invoke({'topic': 'black hole'})

"Here's a 5-point summary of the report on black holes:\n\n1. **Formation and Properties of Black Holes**: Black holes are formed when a massive star collapses in on itself at the end of its life, creating an intense gravitational field that warps the fabric of spacetime. They have distinct properties, including mass, event horizon, singularity, and ergosphere.\n\n2. **Types of Black Holes**: There are four types of black holes: stellar black holes (formed from individual stars), intermediate-mass black holes (with masses between stellar-mass and supermassive black holes), supermassive black holes (found at galaxy centers, with masses millions or billions of times that of the sun), and primordial black holes (hypothetical, formed in the early universe).\n\n3. **Detection Methods**: Black holes are difficult to detect directly, but astronomers use indirect methods, such as detecting X-rays and gamma rays from hot gas, radio waves from matter spiraling in, gravitational waves, and observ

You can observe that the StrOutputParser is a much easier and cleaner way of doing the code than manual LLM parsing

### **JSONOutputParser**

This is the quickest possible way to extract JSON output from the LLM models

In [25]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

template = PromptTemplate(
    template = "give me the name, age and city of sir arthur dayne in a song of ice and fire\n {format_instruction}",
    input_varaibles = [],
    partial_variables = {'format_instruction': parser.get_format_instructions()}
)

prompt = template.format()

print(prompt)

give me the name, age and city of sir arthur dayne in a song of ice and fire
 Return a JSON object.


We can find that when we add the `` format_instruction`` as a partial variable to the prompt (the JSON Parser), we get the additional line asking to return a JSON output.

This is added before runtime

In [26]:
result = model.invoke(prompt)

print(result)

content='```json\n{\n  "name": "Ser Arthur Dayne, the Sword of the Morning",\n  "age": "Unknown, but estimated to be around 40-50 years old",\n  "city": "Dorne", \n  "affiliation": "The Kingsguard"\n}\n```\nNote: Ser Arthur Dayne\'s age and city are not explicitly mentioned in the book series, so the provided information is an estimate based on available information.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 62, 'total_tokens': 155, 'completion_time': 0.148742745, 'completion_tokens_details': None, 'prompt_time': 0.003691795, 'prompt_tokens_details': None, 'queue_time': 0.066572753, 'total_time': 0.15243454}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019c9337-c59c-7d11-951e-ba04241cd235-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 62, 'output_tokens': 93, 'total

In [27]:
final_result = parser.parse(result.content)

print(final_result)

{'name': 'Ser Arthur Dayne, the Sword of the Morning', 'age': 'Unknown, but estimated to be around 40-50 years old', 'city': 'Dorne', 'affiliation': 'The Kingsguard'}


In [28]:
type(final_result)

dict

The output we got is indeed the JSON object

We can also get various detailed out of the JSON object as we do as usual.

ex: i want to get the name out

In [29]:
print(final_result['name'])

Ser Arthur Dayne, the Sword of the Morning


We can also use langchain chains to do this.

One disadvantage of JSON Output parser is that it does not enforce any schema. To resolve this, we use the StructuredOutputParser

## **StructuredOutputParser**

**StructuredOutputParser** is an output parser in langchain that helps axtract structured JSON output from LLM responses based on predefined field schemas.

It works by defining a list of fields (ResponseSchema) that the model should return, ensuring that the output follows a structured response.

However in langchain 1.2.x, both BaseModel and StructuredOutputParser have been depricated in support of the PydanticOutputParsers

## **PydanticOutputParser**

**PydanticOutputParser** is a structured output parser which uses Pydantic models to enforce schema validation when processing LLM responses.

Why use PydanticOutputParser?

1. **Strict Schema Enforcement**: Ensures that the LLM responses follow a well defined structure.

2. **Type Safety**: Automatically converts LLM outputs to python objects.

3. **Easy Validation**: Uses pydantic's built in validation to handle incorrect and missing data.

4. **Seemless Integration**: works well with other langchain components.

In [40]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [43]:
class Person(BaseModel):
  name: str = Field(description="name of the person")
  age: int = Field(description="age of the person", gt=18)  # we are setting a constraint that age should be greater than 18
  city: str = Field(description="city of the person")

parser = PydanticOutputParser(pydantic_object=Person)


template = PromptTemplate(
    template = "Generate the name, age and city of a fictional {place} person \n {format_instruction}",
    input_variables=['place'],
    partial_variables={'format_instruction': parser.get_format_instructions()}

)

prompt = template.invoke({'place': 'indian'})

result = model.invoke(prompt)

print(result.content)




```json
{
  "name": "Rohan Jain",
  "age": 27,
  "city": "Mumbai"
}
```

This JSON instance conforms to the given schema. It has the required properties (`name`, `age`, and `city`), and each property is of the correct type (`string` and `integer`). Additionally, the `age` is greater than or equal to 18, which satisfies the `exclusiveMinimum` constraint.


In [44]:
print(prompt)

text='Generate the name, age and city of a fictional indian person \n The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "name of the person", "title": "Name", "type": "string"}, "age": {"description": "age of the person", "exclusiveMinimum": 18, "title": "Age", "type": "integer"}, "city": {"description": "city of the person", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}\n```'


### KEY DIFFERENCE:

Structured Output functions validate the structure during the time of generation itself.

ie: the model generates the responses in that structure.

Output parsers however, take the LLM output as text and then parse them into the struecture we specified.


There is nothing as one better than the other. we use them as per the use case.

**Structured output = compile-time type safety**

**Output parser = runtime validation layer**

**One enforces structure at generation.**
**The other enforces structure after generation.**